# 火灾疏散项目 — Colab 训练脚本
1. 菜单栏 → **修改** → **笔记本设置** → 硬件加速器选 **T4 GPU**
2. 逐个点播放按钮从上到下运行

In [ ]:
# [1] 挂载 Google Drive（可选，取消也不影响）
import os
USE_DRIVE = False
DRIVE_PATH = '/content/drive/MyDrive/huo-zai-checkpoints'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_PATH, exist_ok=True)
    USE_DRIVE = True
    print('Google Drive 已挂载')
except Exception as e:
    print(f'Drive 跳过，模型将通过浏览器下载')

In [ ]:
# [2] 一次性克隆两个仓库（代码 + 数据集）
import os
!rm -rf LonlyStydue Social-STGCNN
!git clone -q https://github.com/confidentismylife/LonlyStydue.git
!git clone -q https://github.com/abduallahmohamed/Social-STGCNN.git
%cd LonlyStydue
!pip install -q torch numpy pandas matplotlib scipy scikit-learn tqdm
print('代码 + 数据集就绪')

In [ ]:
# [3] 验证数据文件
import os
data_dir = '/content/Social-STGCNN/datasets'
txt_files = []
for root, dirs, files in os.walk(data_dir):
    for f in files:
        if f.endswith('.txt'):
            txt_files.append(os.path.join(root, f))
print(f'找到 {len(txt_files)} 个数据文件')

# 看一个样本确认格式
if txt_files:
    with open(txt_files[0], 'r') as f:
        print(f'样本 ({os.path.basename(txt_files[0])}):')
        for i, line in enumerate(f):
            if i >= 3: break
            print(f'  {repr(line.strip())}')

In [ ]:
# [4] 检查 GPU
import torch
print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'型号: {torch.cuda.get_device_name(0)}')

In [ ]:
# [5] 加载数据（格式已验证: frame_id ped_id x y，tab 分隔）
import numpy as np
from collections import defaultdict

OBS_LEN, PRED_LEN = 8, 12

def load_file(filepath):
    data = defaultdict(list)
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().replace('\t', ' ').split()
            if len(parts) < 4:
                continue
            try:
                frame, pid = int(float(parts[0])), int(float(parts[1]))
                x, y = float(parts[2]), float(parts[3])
                data[pid].append((frame, x, y))
            except:
                continue
    return data

def extract(data):
    obs_list, pred_list = [], []
    for pid, traj in data.items():
        traj.sort(key=lambda t: t[0])
        for i in range(len(traj) - OBS_LEN - PRED_LEN + 1):
            seg = traj[i:i+OBS_LEN+PRED_LEN]
            obs = np.array([[t[1], t[2]] for t in seg[:OBS_LEN]], dtype=np.float32)
            pred = np.array([[t[1], t[2]] for t in seg[OBS_LEN:]], dtype=np.float32)
            obs_list.append(obs)
            pred_list.append(pred)
    return obs_list, pred_list

all_obs, all_preds = [], []
for fpath in txt_files:
    raw = load_file(fpath)
    obs, preds = extract(raw)
    if obs:
        all_obs.extend(obs)
        all_preds.extend(preds)

all_obs = np.array(all_obs)
all_preds = np.array(all_preds)
print(f'轨迹数: {len(all_obs):,}  |  obs: {all_obs.shape}  |  pred: {all_preds.shape}')

In [ ]:
# [6] 归一化 + 划分
origin = all_obs[:, -1:, :]
all_obs_n = all_obs - origin
all_preds_n = all_preds - origin

np.random.seed(42)
idx = np.random.permutation(len(all_obs_n))
n_train = int(len(idx) * 0.8)
train_obs = all_obs_n[idx[:n_train]]
train_preds = all_preds_n[idx[:n_train]]
val_obs = all_obs_n[idx[n_train:]]
val_preds = all_preds_n[idx[n_train:]]
print(f'训练: {len(train_obs):,}  |  验证: {len(val_obs):,}')

In [ ]:
# [7] 训练
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from src.model import SimpleLSTM
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class TD(Dataset):
    def __init__(self, o, p):
        self.o = torch.from_numpy(o).float()
        self.p = torch.from_numpy(p).float()
    def __len__(self): return len(self.o)
    def __getitem__(self, i): return self.o[i], self.p[i]

tl = DataLoader(TD(train_obs, train_preds), batch_size=128, shuffle=True)
vl = DataLoader(TD(val_obs, val_preds), batch_size=128, shuffle=False)

model = SimpleLSTM(hidden_dim=64, pred_len=PRED_LEN).to(device)
print(f'参数: {sum(p.numel() for p in model.parameters()):,}  |  {device}')

opt = torch.optim.Adam(model.parameters(), lr=0.001)
sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=10)
crit = nn.MSELoss()

best_ade = float('inf')
for ep in range(1, 101):
    t0 = time.time()
    model.train()
    loss_sum = 0.0
    for o, p in tl:
        o, p = o.to(device), p.to(device)
        opt.zero_grad()
        L = crit(model(o), p)
        L.backward()
        opt.step()
        loss_sum += L.item() * o.size(0)
    model.eval()
    ade = fde = n = 0.0
    with torch.no_grad():
        for o, p in vl:
            o, p = o.to(device), p.to(device)
            d = model(o) - p
            ade += torch.norm(d, dim=2).mean(1).sum().item()
            fde += torch.norm(d[:,-1,:], dim=1).sum().item()
            n += o.size(0)
    vade, vfde = ade/n, fde/n
    sch.step(vade)
    if vade < best_ade:
        best_ade = vade
        os.makedirs('checkpoints', exist_ok=True)
        torch.save(model.state_dict(), 'checkpoints/best_model.pth')
    if ep <= 3 or ep % 20 == 0:
        print(f'Epoch {ep:3d} | Loss {loss_sum/n:.4f} | ADE {vade:.3f}m | FDE {vfde:.3f}m | {time.time()-t0:.1f}s')

print(f'\n最佳 ADE: {best_ade:.3f}m  (基线=0.48m)')

In [ ]:
# [8] 下载模型
if USE_DRIVE:
    import shutil
    shutil.copy('checkpoints/best_model.pth', os.path.join(DRIVE_PATH, 'best_model.pth'))
    print('已存到 Google Drive')
else:
    from google.colab import files
    files.download('checkpoints/best_model.pth')
    print('下载后放到 Huo-zai/checkpoints/ 下')

In [ ]:
# [9] 可视化 3 条预测结果
import matplotlib.pyplot as plt
model.eval()
ids = np.random.choice(len(val_obs), 3, replace=False)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, idx in enumerate(ids):
    obs_i = torch.from_numpy(val_obs[idx]).float().unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(obs_i).squeeze(0).cpu().numpy()
    obs_i = obs_i.squeeze(0).cpu().numpy()
    gt = val_preds[idx]
    ax = axes[i]
    ax.plot(obs_i[:,0], obs_i[:,1], 'b-o', label='Obs')
    ax.plot(pred[:,0], pred[:,1], 'r--o', label='Pred')
    ax.plot(gt[:,0], gt[:,1], 'g-o', alpha=0.5, label='GT')
    ax.set_title(f'Sample {i+1}'); ax.set_aspect('equal')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('results.png', dpi=100); plt.show()